In [1]:
# Instalação da biblioteca necessária (caso não esteja no ambiente padrão)
!pip install ortools pandas -q

In [2]:
import pandas as pd
from ortools.linear_solver import pywraplp

In [3]:
# Carregar o arquivo CSV
df = pd.read_csv('tempos.csv')

# Definir conjuntos de pessoas e tarefas baseados no enunciado
pessoas = range(1, 21) # P = {1, 2, ..., 20}
tarefas = range(1, 21) # T = {1, 2, ..., 20}

# Criar dicionário de custos para mapear o tempo que a pessoa i leva para a tarefa j
custos = {}
for _, row in df.iterrows():
    custos[int(row['id_pessoa']), int(row['id_tarefa'])] = row['tempo']

print(f"Dados carregados: {len(custos)} combinações de tempos lidas.")

Dados carregados: 400 combinações de tempos lidas.


In [4]:
# Criar o solver utilizando o backend SCIP
solver = pywraplp.Solver.CreateSolver('SCIP')

# 1. Variáveis de decisão: x[i, j] é 1 se a pessoa i for designada para a tarefa j
x = {}
for i in pessoas:
    for j in tarefas:
        x[i, j] = solver.IntVar(0, 1, f'x_{i}_{j}')

# 2. Restrição: Cada pessoa deve executar exatamente uma tarefa
for i in pessoas:
    solver.Add(solver.Sum([x[i, j] for j in tarefas]) == 1)

# 3. Restrição: Cada tarefa deve ser executada por exatamente uma pessoa
for j in tarefas:
    solver.Add(solver.Sum([x[i, j] for i in pessoas]) == 1)

# 4. Objetivo: Minimizar o tempo total de execução
objetivo = solver.Sum([custos[i, j] * x[i, j] for i in pessoas for j in tarefas])
solver.Minimize(objetivo)

# Resolver o problema
status = solver.Solve()

In [ ]:
if status == pywraplp.Solver.OPTIMAL:
    print(f'Tempo Total de Execução (Mínimo): {solver.Objective().Value()}\n')
    
    atribuicoes = []
    for i in pessoas:
        for j in tarefas:
            if x[i, j].solution_value() > 0.5:
                atribuicoes.append({
                    'Pessoa': i,
                    'Tarefa': j,
                    'Tempo': custos[i, j]
                })
    
    # Exibir resultado em formato de tabela
    df_resultado = pd.DataFrame(atribuicoes)
    print(df_resultado.to_string(index=False))
else:
    print('Não foi encontrada uma solução ótima para o problema.')
    # comentario

Tempo Total de Execução (Mínimo): 134.0

 Pessoa  Tarefa  Tempo
      1       4      2
      2      16      9
      3      14      1
      4      17      5
      5      12      9
      6       1     13
      7       9      8
      8       6      3
      9      18      1
     10      19     13
     11      15      1
     12       3      5
     13       8      6
     14       5      5
     15      10      3
     16      13     19
     17       7      4
     18      20      3
     19      11     13
     20       2     11
